# Gemma 4 Cookbook

**Model Family:** Google DeepMind's Gemma 4  
**Latest Update:** April 2026  
**License:** Apache-2.0  
**Provider:** Google DeepMind

Gemma 4 is Google's latest family of open-source multimodal models with hybrid-thinking capabilities, supporting 140+ languages, up to 256K context windows, and both dense and Mixture-of-Experts (MoE) variants. The models excel at reasoning, coding, tool use, long-context processing, and agentic workflows.

**Model Variants:**
- **E2B/E4B**: Lightweight models designed for phones and laptops (5GB RAM at 4-bit)
- **26B-A4B**: MoE model balancing speed and accuracy with 4B active parameters (18GB RAM at 4-bit)
- **31B**: The strongest variant for maximum quality (20GB RAM at 4-bit)

**Key Features:**
- Input modalities: Text, Images, Audio (E2B/E4B), Video frames
- Multimodal: Supports visual understanding, OCR, document parsing
- Context window: Up to 256K tokens
- Languages: 140+ language support
- Hybrid thinking: Explicit reasoning channel with `<|think|>` token control
- Apache-2.0 licensed: Can be run and fine-tuned locally

**Best Use Cases:**
- Reasoning and complex problem-solving
- Code generation and debugging
- Multimodal understanding (images, audio, text)
- Long document processing and analysis
- Agentic workflows with tool calling
- Audio transcription and translation (E2B/E4B)

**Table of Contents:**
1. Setup and Installation
2. Model Loading and Configuration
3. Hardware Requirements and Inference
4. Text Generation and Prompting
5. Thinking Mode and Reasoning
6. Multimodal Capabilities
7. Recommended Settings and Best Practices
8. Production Deployment

## 1. Setup and Installation

Install required packages for running Gemma 4 models. Choose your approach based on your hardware and use case.

In [ ]:
import subprocess
import sys

# Install core dependencies
packages = [
    "transformers>=4.44.0",
    "torch>=2.1.0",
    "numpy",
    "requests",
]

def install_packages(packages_list):
    """Install required packages"""
    install_cmd = [sys.executable, "-m", "pip", "install", "-q"] + packages_list
    result = subprocess.run(install_cmd, capture_output=True, text=True)
    return result.returncode == 0

if install_packages(packages):
    print("✓ Core packages installed successfully")
    print("  - transformers (for model loading)")
    print("  - torch (deep learning framework)")
    print("  - numpy (numerical computing)")
else:
    print("⚠️  Some packages may have failed to install")

# Optional packages for specific use cases:
# For quantization (llama.cpp GGUF format): pip install llama-cpp-python
# For GPTQ quantization: pip install auto-gptq
# For MLX support (Apple Silicon): pip install mlx
# For vLLM serving: pip install vllm
# For ray distributed inference: pip install ray

## 2. Hardware Requirements and Model Selection

### Memory Requirements (in total RAM/VRAM)

| Model | 4-bit | 8-bit | BF16/FP16 |
|-------|-------|-------|-----------|
| **E2B** | 5 GB | 10 GB | 16 GB |
| **E4B** | 6 GB | 15 GB | 28 GB |
| **26B-A4B** | 18 GB | 28 GB | 56 GB |
| **31B** | 20 GB | 34 GB | 68 GB |

**Supported Hardware Platforms:**
- NVIDIA CUDA-enabled GPUs (RTX, A-series, H-series)
- Apple Metal (M1/M2/M3 chips) with native quantization support
- CPU inference (slower, but possible with quantization)
- Cloud TPUs and other accelerators via compatible frameworks

**Model Selection Guide:**

- **E2B / E4B**: Mobile and laptop deployment, real-time local inference with minimal resources
- **26B-A4B**: Recommended for balanced speed and quality on consumer GPUs with MoE efficiency
- **31B**: Maximum quality and reasoning capability when sufficient VRAM is available

### As a rule of thumb:
Your available memory should exceed the quantized model size. If not, dynamic RAM/disk offload is possible but will reduce generation speed.

## 3. Loading Gemma 4 Models

### Option 1: Standard Transformers (Safetensor Format)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Model identifiers available on Hugging Face Hub
GEMMA_4_MODELS = {
    "e2b": "google/gemma-4-e2b-instruct",
    "e4b": "google/gemma-4-e4b-instruct", 
    "26b-a4b": "google/gemma-4-26b-a4b-it",
    "31b": "google/gemma-4-31b-it"
}

def load_model_and_tokenizer(model_name="26b-a4b", quantize=True):
    """
    Load Gemma 4 model and tokenizer
    
    Args:
        model_name: Model variant ('e2b', 'e4b', '26b-a4b', '31b')
        quantize: Use 4-bit quantization (requires bitsandbytes)
    
    Returns:
        Tuple of (model, tokenizer)
    """
    model_id = GEMMA_4_MODELS.get(model_name, GEMMA_4_MODELS["26b-a4b"])
    
    print(f"Loading {model_name} from Hugging Face...")
    
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    
    # Load model with optional 4-bit quantization
    if quantize:
        try:
            from transformers import BitsAndBytesConfig
            
            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.bfloat16,
                bnb_4bit_use_double_quant=True,
            )
            model = AutoModelForCausalLM.from_pretrained(
                model_id,
                quantization_config=bnb_config,
                device_map="auto"
            )
            print(f"✓ Loaded {model_name} with 4-bit quantization")
        except ImportError:
            print("⚠️  bitsandbytes not installed, loading without quantization")
            model = AutoModelForCausalLM.from_pretrained(
                model_id,
                torch_dtype=torch.bfloat16,
                device_map="auto"
            )
    else:
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch.bfloat16,
            device_map="auto"
        )
        print(f"✓ Loaded {model_name} (unquantized)")
    
    return model, tokenizer

# Uncomment to load a model:
# model, tokenizer = load_model_and_tokenizer("26b-a4b", quantize=True)

### Option 2: GGUF Format (Quantized with llama.cpp)

For CPU inference or extreme quantization, use GGUF format models available on Hugging Face Hub. Quantization types include:
- `UD-Q4_K_XL`: Dynamic 4-bit quantization for 26B-A4B and 31B (recommended)
- `Q8_0`: 8-bit quantization for E2B and E4B
- `UD-Q2_K_XL`: Dynamic 2-bit quantization for smallest footprint

In [ ]:
# Download GGUF model example (requires huggingface_hub)
# import subprocess
# subprocess.run([
#     "pip", "install", "huggingface-hub[hf-transfer]"
# ])

# from huggingface_hub import hf_download
# 
# # Download 26B-A4B model with 4-bit quantization
# hf_download(
#     repo_id="google/gemma-4-26b-a4b-it-GGUF",
#     local_dir="./gemma-4-26b-a4b-it-GGUF",
#     include=["*UD-Q4_K_XL*", "*mmproj*"]
# )

## 4. Text Generation and Inference

### Recommended Inference Parameters

Gemma 4 uses industry-standard inference parameters. Start with these defaults:

- **Temperature**: 1.0 (higher = more creative, lower = more deterministic)
- **Top-p (nucleus sampling)**: 0.95 (consider tokens with cumulative probability up to 0.95)
- **Top-k**: 64 (consider only top 64 tokens at each step)
- **Context window**: Start with 32K, then increase as needed (up to 256K)
- **Repetition penalty**: 1.0 (unless experiencing looping, then increase slightly)
- **End of Sentence token**: `<turn|>`

## 5. Hybrid Thinking Mode

Gemma 4 includes a powerful thinking mechanism that enables the model to show its internal reasoning. This is controlled using the `<|think|>` token.

### How to Enable Thinking:

Add the token `<|think|>` at the start of your system prompt. When thinking is enabled:
- The model outputs an internal reasoning channel before the final answer
- Output format: `<|channel>thought[reasoning]<channel|>[final answer]<turn|>`

### Example: Thinking Enabled vs Disabled

In [ ]:
def generate_with_thinking(model, tokenizer, prompt, max_tokens=1024, use_thinking=True):
    """
    Generate text with optional thinking mode
    
    Args:
        model: Loaded Gemma 4 model
        tokenizer: Associated tokenizer
        prompt: Input prompt
        max_tokens: Maximum tokens to generate
        use_thinking: Enable thinking mode
    
    Returns:
        Generated text
    """
    
    # System prompt configuration
    system_prompt = "<|think|>" if use_thinking else ""
    
    system_prompt += "You are a precise reasoning assistant. Explain your answer clearly."
    
    # Format with proper chat template
    # Standard Gemma 4 format uses: system, assistant, and user roles
    conversation = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]
    
    # Apply chat template
    text = tokenizer.apply_chat_template(
        conversation,
        tokenize=False,
        add_generation_prompt=True
    )
    
    # Tokenize
    inputs = tokenizer.encode(text, return_tensors="pt").to(model.device)
    
    # Generate with specified parameters
    outputs = model.generate(
        inputs,
        max_new_tokens=max_tokens,
        temperature=1.0,
        top_p=0.95,
        top_k=64,
        do_sample=True,
    )
    
    # Decode and return
    return tokenizer.decode(outputs[0], skip_special_tokens=False)

# Example usage (uncomment when model is loaded):
# prompt = "A train leaves at 8:15 AM and arrives at 11:47 AM. How long was the journey?"
# result = generate_with_thinking(model, tokenizer, prompt, use_thinking=True)
# print("With thinking:")
# print(result)
# print("\nWithout thinking:")
# result_no_think = generate_with_thinking(model, tokenizer, prompt, use_thinking=False)
# print(result_no_think)

## 6. Multimodal Capabilities

### Supported Modalities by Model:

| Feature | E2B | E4B | 26B-A4B | 31B |
|---------|-----|-----|---------|-----|
| Text | ✓ | ✓ | ✓ | ✓ |
| Vision | ✓ | ✓ | ✓ | ✓ |
| Audio | ✓ | ✓ | ✗ | ✗ |
| Video | ✓ | ✓ | ✓ | ✓ |

### Visual Token Budgets:

Gemma 4 supports multiple visual token budgets for different use cases:

- **70 tokens**: Fast classification, image captioning, quick video understanding
- **140 tokens**: General chat and quick analysis
- **280 tokens**: Standard multimodal chat, charts, screens, UI reasoning  
- **560 tokens**: Detailed analysis, complex layouts
- **1120 tokens**: OCR, document parsing, handwriting, small text extraction

### Best Practices for Multimodal Prompting:

1. **Content order**: Always place visual/audio content FIRST, then text instructions
2. **Video handling**: Provide as frame sequence (1 frame per second), then instruction
3. **Audio limits**: Maximum 30 seconds for audio input
4. **Video limits**: Maximum 60 seconds at 1 frame per second

### Example Prompts for Different Tasks

#### Vision - OCR and Document Parsing:
```
[image first]
Extract all text from this document. Return as JSON with fields: headers, body, tables, total_amount.
Use visual token budget of 1120 for best results.
```

#### Vision - UI/Screen Analysis:
```
[screenshot]
Analyze this user interface. Which elements might confuse new users and why?
Use visual token budget of 280 for this task.
```

#### Audio - Speech Recognition (E2B/E4B only):
```
[audio first]
Transcribe the following speech in English. Output only the transcription with no newlines.
When transcribing numbers, write digits (1.7 not one point seven).
```

#### Audio - Translation (E2B/E4B only):
```
[audio first]
Transcribe the speech in Spanish, then translate to English.
Format: First line = Spanish transcription, Second line = "English: " + translation
```

#### Video Understanding:
```
[video frames sequence]
Describe what's happening in this video. Focus on: key actions, people involved, and timeline.
```

## 7. Prompting Examples and Best Practices

### Simple Reasoning Prompt:
```
System:
<|think|>
You are a precise reasoning assistant.

User:
A train leaves at 8:15 AM and arrives at 11:47 AM. How long was the journey?
```

### Code Generation Prompt:
```
System:
You are an expert Python developer. Write clean, well-documented code.

User:
Write a function that checks if a string is a palindrome, ignoring spaces and punctuation.
```

### Multi-turn Conversation Tips:

- For multi-turn conversations, only keep the final visible answer in chat history
- Do NOT feed prior `<|channel>thought` blocks back into the next turn
- This prevents confusion and maintains conversation flow
- The model can attend to full context efficiently up to 256K tokens

## 8. Fine-tuning for Custom Tasks

### Supervised Fine-Tuning (SFT) Setup

Fine-tune Gemma 4 on your own data using standard supervised fine-tuning approaches.

In [ ]:
from transformers import TrainingArguments, Trainer
from datasets import load_dataset

def fine_tune_gemma(
    model,
    tokenizer,
    dataset_path,
    output_dir="gemma-4-finetuned",
    num_epochs=3,
    batch_size=4,
):
    """
    Fine-tune Gemma 4 on custom dataset
    
    Args:
        model: Loaded Gemma 4 model
        tokenizer: Associated tokenizer
        dataset_path: Path to training dataset (JSON/CSV with 'text' or 'instruction'/'input'/'output' columns)
        output_dir: Output directory for checkpoints
        num_epochs: Number of training epochs
        batch_size: Batch size for training
    """
    
    # Load dataset
    dataset = load_dataset("json", data_files=dataset_path)
    
    def preprocess_function(examples):
        """Tokenize examples"""
        return tokenizer(
            examples["text"],
            truncation=True,
            max_length=2048,
            padding="max_length"
        )
    
    # Preprocess dataset
    tokenized_dataset = dataset.map(
        preprocess_function,
        batched=True,
        remove_columns=["text"]
    )
    
    # Training arguments
    training_args = TrainingArguments(
        output_dir=output_dir,
        overwrite_output_dir=True,
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,
        save_steps=100,
        save_total_limit=3,
        logging_steps=10,
        learning_rate=2e-4,
        bf16=True,  # Use bfloat16 for memory efficiency
        gradient_accumulation_steps=2,
    )
    
    # Create trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset["train"],
    )
    
    # Fine-tune
    trainer.train()
    
    # Save fine-tuned model
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"✓ Fine-tuned model saved to {output_dir}")
    
    return model, tokenizer

# Example dataset format (JSON):
# [
#   {"text": "User: How are you?\nAssistant: I'm doing well, thank you!"},
#   {"text": "User: What's 2+2?\nAssistant: 2+2 equals 4"}
# ]

# To fine-tune (uncomment when ready):
# fine_tune_gemma(model, tokenizer, "training_data.json")

## 9. Optimization Techniques

### Memory Optimization

**Quantization Methods:**

1. **4-bit Quantization** (Recommended for most cases)
   - Reduces model from 68GB (32B, FP16) to ~20GB
   - Uses bitsandbytes or GPTQ
   - Minimal quality loss
   
2. **8-bit Quantization**
   - Middle ground between quality and size
   - Good for E4B models (15GB → ~8GB)

3. **GGUF Format (llama.cpp)**
   - CPU-optimized quantization
   - Dynamic 4-bit or 2-bit options
   - Best for edge devices

### Speed Optimization:

- Use `flash-attention-2` if available
- Batch inference when possible
- Use KV-cache for multi-turn conversations
- Reduce context window if full 256K not needed

In [ ]:
def batch_generate(model, tokenizer, prompts, batch_size=8, max_tokens=256):
    """
    Efficient batch inference for multiple prompts
    
    Args:
        model: Loaded Gemma 4 model
        tokenizer: Associated tokenizer
        prompts: List of prompts to generate for
        batch_size: Number of prompts to process together
        max_tokens: Maximum tokens per generation
    
    Returns:
        List of generated texts
    """
    results = []
    
    for i in range(0, len(prompts), batch_size):
        batch_prompts = prompts[i:i + batch_size]
        
        # Tokenize batch
        inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=2048
        ).to(model.device)
        
        # Generate batch
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_tokens,
                temperature=1.0,
                top_p=0.95,
                top_k=64,
                do_sample=True,
            )
        
        # Decode batch
        batch_results = tokenizer.batch_decode(
            outputs,
            skip_special_tokens=True
        )
        results.extend(batch_results)
    
    return results

# Example usage:
# prompts = [
#     "What is machine learning?",
#     "Explain quantum computing",
#     "How do neural networks work?"
# ]
# results = batch_generate(model, tokenizer, prompts)

## 10. Production Deployment Patterns

### REST API Server Example

For production use, wrap your model in a REST API using FastAPI or similar frameworks.

In [ ]:
# Example FastAPI server for Gemma 4
# Install: pip install fastapi uvicorn

"""
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

app = FastAPI(title="Gemma 4 API")

# Load model on startup
model = None
tokenizer = None

@app.on_event("startup")
async def load_model():
    global model, tokenizer
    model_id = "google/gemma-4-26b-a4b-it"
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16,
        device_map="auto"
    )

class GenerationRequest(BaseModel):
    prompt: str
    max_tokens: int = 256
    temperature: float = 1.0
    thinking: bool = False

class GenerationResponse(BaseModel):
    generated_text: str
    tokens_used: int

@app.post("/generate", response_model=GenerationResponse)
async def generate(request: GenerationRequest):
    if model is None:
        raise HTTPException(status_code=503, detail="Model not loaded")
    
    try:
        system_prompt = "<|think|>" if request.thinking else ""
        system_prompt += "You are a helpful assistant."
        
        conversation = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": request.prompt}
        ]
        
        text = tokenizer.apply_chat_template(
            conversation,
            tokenize=False,
            add_generation_prompt=True
        )
        
        inputs = tokenizer.encode(text, return_tensors="pt").to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                inputs,
                max_new_tokens=request.max_tokens,
                temperature=request.temperature,
                top_p=0.95,
                top_k=64,
                do_sample=True,
            )
        
        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        return GenerationResponse(
            generated_text=generated_text,
            tokens_used=len(outputs[0])
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# Run with: uvicorn script_name:app --host 0.0.0.0 --port 8000
"""

print("FastAPI example server code above. Uncomment to use.")

## 11. Quick Reference and Best Practices

### Model Selection Quick Guide

```
╔════════════════╦════════════╦════════════╦═══════════╗
║ Use Case       ║ Model      ║ RAM (4-bit)║ Speed     ║
╠════════════════╬════════════╬════════════╬═══════════╣
║ Mobile/Laptop  ║ E2B / E4B  ║ 5-6 GB     ║ Fastest   ║
║ Quality/Speed  ║ 26B-A4B    ║ 18 GB      ║ Fast      ║
║ Best Quality   ║ 31B        ║ 20 GB      ║ Standard  ║
╚════════════════╩════════════╩════════════╩═══════════╝
```

### Inference Parameters Quick Reference

```python
# Default Gemma 4 parameters
params = {
    "temperature": 1.0,
    "top_p": 0.95,
    "top_k": 64,
    "max_context": 32768,  # Start here, scale to 256K as needed
    "repetition_penalty": 1.0,
    "eos_token": "<turn|>"
}
```

### Thinking Mode Decision Tree

```
Do you need explicit reasoning?
├─ YES: Add <|think|> to system prompt
│       - Output will show <|channel>thought...<channel|>
│       - Slower but more transparent
└─ NO: Skip <|think|> token
       - Faster generation
       - Still capable of reasoning internally
```

### Common Issues and Solutions

| Issue | Solution |
|-------|----------|
| Out of memory | Use 4-bit quantization or smaller model (E2B) |
| Slow generation | Reduce context window, increase batch size |
| Model looping | Increase repetition_penalty to 1.1-1.3 |
| Low quality | Increase model size (26B-A4B or 31B) or improve prompt |
| CUDA issues | Update pytorch: `pip install -U torch` |

### Performance Benchmarks

On standard hardware (RTX 4090):
- **E2B**: ~50-100 tokens/sec
- **E4B**: ~40-80 tokens/sec  
- **26B-A4B**: ~30-50 tokens/sec
- **31B**: ~20-40 tokens/sec

(Speeds vary based on quantization, batch size, and context length)

## 12. Additional Resources and References

### Official Documentation & Blog Posts

- **Hugging Face Gemma 4 Blog**: Latest announcements and technical details
- **NVIDIA Blog**: GPU optimization and deployment best practices
- **Google DeepMind Official Blog**: Model release information and capabilities

### Model Repositories

- **Gemma 4 on Hugging Face**: Full model collection with safetensor and GGUF formats
- **Transformers Library**: Canonical implementation and loading utilities

### Related Technologies

**Inference Frameworks:**
- `llama.cpp`: Fast CPU and GPU inference with GGUF quantization
- `vLLM`: Efficient serving framework with batching support
- `TGI` (Text Generation Inference): Optimized for production workloads

**Quantization Tools:**
- `bitsandbytes`: 4-bit and 8-bit quantization
- `GPTQ`: GPU-optimized quantization
- `AWQ`: Activation-aware quantization

**Fine-Tuning Approaches:**
- Supervised Fine-Tuning (SFT): Standard approach for instruction tuning
- LoRA (Low-Rank Adaptation): Memory-efficient fine-tuning
- QLoRA: Combined quantization + LoRA for extreme efficiency

### Community Resources

- GitHub repositories for implementations
- Discussion forums for troubleshooting
- Papers on reasoning models and thinking mechanisms

---

**Last Updated:** April 2026  
**Gemma 4 Family Version:** Latest stable release  
**License:** Models released under Apache-2.0 license

## 13. Complete Practical Example

Here's a complete example combining model loading, configuration, and inference with thinking mode:

In [ ]:
#!/usr/bin/env python3
"""
Complete Gemma 4 Example: Load, Configure, and Inference
"""

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

class Gemma4Processor:
    """Complete Gemma 4 processor with all features"""
    
    def __init__(self, model_variant="26b-a4b", quantize=True):
        self.variant = model_variant
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = None
        self.tokenizer = None
        self.load_model(quantize)
    
    def load_model(self, quantize=True):
        """Load model and tokenizer"""
        models = {
            "e2b": "google/gemma-4-e2b-instruct",
            "e4b": "google/gemma-4-e4b-instruct",
            "26b-a4b": "google/gemma-4-26b-a4b-it",
            "31b": "google/gemma-4-31b-it"
        }
        
        model_id = models.get(self.variant, models["26b-a4b"])
        
        print(f"📦 Loading {self.variant}...")
        
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        
        if quantize and self.device == "cuda":
            from transformers import BitsAndBytesConfig
            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.bfloat16,
                bnb_4bit_use_double_quant=True,
            )
            self.model = AutoModelForCausalLM.from_pretrained(
                model_id, quantization_config=bnb_config, device_map="auto"
            )
        else:
            self.model = AutoModelForCausalLM.from_pretrained(
                model_id, torch_dtype=torch.bfloat16, device_map="auto"
            )
        
        print(f"✓ Model loaded on {self.device}")
    
    def generate(self, prompt, use_thinking=True, max_tokens=512):
        """Generate response with optional thinking"""
        
        system = "<|think|>" if use_thinking else ""
        system += "You are a helpful and precise assistant."
        
        conversation = [
            {"role": "system", "content": system},
            {"role": "user", "content": prompt}
        ]
        
        text = self.tokenizer.apply_chat_template(
            conversation, tokenize=False, add_generation_prompt=True
        )
        
        inputs = self.tokenizer.encode(text, return_tensors="pt").to(self.device)
        
        with torch.no_grad():
            outputs = self.model.generate(
                inputs,
                max_new_tokens=max_tokens,
                temperature=1.0,
                top_p=0.95,
                top_k=64,
                do_sample=True,
            )
        
        return self.tokenizer.decode(outputs[0], skip_special_tokens=False)

# Example usage:
if __name__ == "__main__":
    # Initialize processor
    processor = Gemma4Processor(model_variant="26b-a4b", quantize=True)
    
    # Test 1: Simple question without thinking
    print("\n" + "="*60)
    print("Test 1: Simple Question (No Thinking)")
    print("="*60)
    response = processor.generate(
        "What is 2+2?",
        use_thinking=False,
        max_tokens=256
    )
    print(response)
    
    # Test 2: Complex reasoning with thinking
    print("\n" + "="*60)
    print("Test 2: Complex Reasoning (With Thinking)")
    print("="*60)
    response = processor.generate(
        "A store has 15 apples. They receive 8 more apples and sell 12. How many apples do they have now?",
        use_thinking=True,
        max_tokens=512
    )
    print(response)
    
    # Test 3: Code generation
    print("\n" + "="*60)
    print("Test 3: Code Generation")
    print("="*60)
    response = processor.generate(
        "Write a Python function that reverses a string",
        use_thinking=False,
        max_tokens=256
    )
    print(response)